In [11]:
import shutil
import os, cv2
import numpy as np
import pandas as pd

from PIL import Image
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, as_completed
from albumentations import (Compose, HorizontalFlip, RandomBrightnessContrast,
                            Rotate, Affine, GaussianBlur, GaussNoise,
                            RandomGamma, OpticalDistortion, Perspective, Equalize)

In [2]:
WORKSPACE = "/home/admins/rebuild_workspace"
GRID_DIR  = os.path.join(WORKSPACE, "04_grids_combined")
ASPECT_DIR = os.path.join(WORKSPACE, "05b_grids_aspect_preserved")
LOGS_DIR  = os.path.join(WORKSPACE, "logs")

NEW_SIZE = (224, 267)      # (width, height) — 672*0.333, 800*0.333
os.makedirs(ASPECT_DIR, exist_ok=True)

def resize_one(fname):
    with Image.open(os.path.join(GRID_DIR, fname)) as img:
        orig = img.size
        img.resize(NEW_SIZE, Image.LANCZOS).save(os.path.join(ASPECT_DIR, fname))
    return dict(filename=fname, orig_w=orig[0], orig_h=orig[1],
                new_w=NEW_SIZE[0], new_h=NEW_SIZE[1])

files = sorted(f for f in os.listdir(GRID_DIR) if f.endswith('.png'))
print("images:", len(files))

results = []
with ProcessPoolExecutor(max_workers=10) as ex:
    for fut in as_completed([ex.submit(resize_one, f) for f in files]):
        results.append(fut.result())

pd.DataFrame(results).to_csv(f"{LOGS_DIR}/resize_aspect_preserved_log.csv", index=False)
print("written:", len(os.listdir(ASPECT_DIR)))

images: 1300
written: 1300


In [4]:
WORKSPACE  = "/home/admins/rebuild_workspace"
ASPECT_DIR = os.path.join(WORKSPACE, "05b_grids_aspect_preserved")
FINAL_ASPECT = os.path.join(WORKSPACE, "06b_dataset_aspect_preserved")
LOGS_DIR   = os.path.join(WORKSPACE, "logs")

SPLIT_FOLDER = {"train": "Train", "validation": "Validation", "test": "Test"}

mapping = pd.read_csv(f"{LOGS_DIR}/final_image_mapping_log.csv")

for _, r in mapping.iterrows():
    dst = os.path.join(FINAL_ASPECT, SPLIT_FOLDER[r.split], r.word)
    os.makedirs(dst, exist_ok=True)
    shutil.copy2(os.path.join(ASPECT_DIR, r.source_file),
                 os.path.join(dst, r.final_name))

print("placed:", len(mapping))
for sp, folder in SPLIT_FOLDER.items():
    p = os.path.join(FINAL_ASPECT, folder)
    print(f"{folder}:", {w: len(os.listdir(os.path.join(p, w)))
                         for w in sorted(os.listdir(p))})

placed: 1300
Train: {'bat': 90, 'cup': 90, 'drop': 90, 'eat': 90, 'fish': 90, 'hot': 90, 'jump': 90, 'milk': 90, 'pen': 90, 'red': 90}
Validation: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}
Test: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}


In [5]:
REAL_ASPECT = os.path.join(WORKSPACE, "06b_test_real_only_aspect")
words = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']

for word in words:
    dst = os.path.join(REAL_ASPECT, word)
    os.makedirs(dst, exist_ok=True)
    src = os.path.join(FINAL_ASPECT, "Test", word)
    for i in range(1, 10):
        shutil.copy2(os.path.join(src, f"{i:03d}.png"), os.path.join(dst, f"{i:03d}.png"))

print("real-only:", sum(len(os.listdir(os.path.join(REAL_ASPECT, w))) for w in words))

real-only: 90


In [7]:
LOGS_DIR = "/home/admins/rebuild_workspace/logs"

plan_rgb = pd.read_csv(f"{LOGS_DIR}/augmentation_plan_log.csv")
plan_gray = plan_rgb.copy()
plan_gray['aug_type'] = plan_gray['aug_type'].replace("RGBShift", "RandomGamma")
plan_gray.to_csv(f"{LOGS_DIR}/augmentation_plan_gray_log.csv", index=False)

print("substituted:", (plan_rgb.aug_type == "RGBShift").sum(), "entries")
print(plan_gray.aug_type.value_counts().to_dict())

substituted: 5 entries
{'Rotate': 5, 'Scale': 5, 'OpticalDistortion': 5, 'GaussianBlur': 5, 'RandomGamma': 5, 'Translate': 5, 'Perspective': 5, 'Contrast': 5, 'GaussNoise': 5, 'Equalize': 4, 'HorizontalFlip': 4, 'Downsample': 4, 'Sharpen': 4, 'TemporalShift': 3, 'Brightness': 3, 'ProcessCroppedImage': 3}


In [9]:
WORKSPACE   = "/home/admins/rebuild_workspace"
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")
GRAY_DIR    = os.path.join(WORKSPACE, "02g_frames_cropped_gray")
WORDS = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']

def gray_one_word(args):
    set_name, word = args
    src = os.path.join(CROPPED_DIR, set_name, word)
    dst = os.path.join(GRAY_DIR, set_name, word)
    os.makedirs(dst, exist_ok=True)
    n = 0
    for f in sorted(os.listdir(src)):
        if not f.endswith('.png'):
            continue
        img = cv2.imread(os.path.join(src, f))
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        cv2.imwrite(os.path.join(dst, f), g)
        n += 1
    return (set_name, word, n)

tasks = [(f"set_{n:02d}", w) for n in range(1, 61) for w in WORDS]
start = datetime.now()
counts = []
with ProcessPoolExecutor(max_workers=10) as ex:
    for fut in as_completed([ex.submit(gray_one_word, t) for t in tasks]):
        counts.append(fut.result())

print("word folders:", len(counts))
print("frames:", sum(c[2] for c in counts))
print("all 60:", all(c[2] == 60 for c in counts))

sample = cv2.imread(os.path.join(GRAY_DIR, "set_01", "bat", "01.png"), cv2.IMREAD_UNCHANGED)
print("sample shape:", sample.shape, "dtype:", sample.dtype)
print("elapsed:", (datetime.now()-start).seconds, "s")

word folders: 600
frames: 36000
all 60: True
sample shape: (80, 112) dtype: uint8
elapsed: 3 s


In [14]:
def get_augmentations_gray(t):
    a = {
        "HorizontalFlip": Compose([HorizontalFlip(p=1.0)]),
        "Brightness": Compose([RandomBrightnessContrast(brightness_limit=(0.3,0.3), contrast_limit=0.0, p=1.0)]),
        "Contrast": Compose([RandomBrightnessContrast(brightness_limit=0.0, contrast_limit=(0.3,0.3), p=1.0)]),
        "Rotate": Compose([Rotate(limit=5, p=1.0)]),
        "Translate": Compose([Affine(translate_px={"x":(-10,10),"y":(-10,10)}, p=1.0)]),
        "Scale": Compose([Affine(scale=(0.9,1.1), p=1.0)]),
        "GaussianBlur": Compose([GaussianBlur(blur_limit=(7,15), p=1.0)]),
        "GaussNoise": Compose([GaussNoise(var_limit=(100.0,250.0), p=1.0)]),
        "RandomGamma": Compose([RandomGamma(gamma_limit=(60,140), p=1.0)]),
        "OpticalDistortion": Compose([OpticalDistortion(distort_limit=0.2, shift_limit=0.2, p=1.0)]),
        "Perspective": Compose([Perspective(scale=(0.05,0.1), p=1.0)]),
        "Equalize": Compose([Equalize(p=1.0)]),
    }
    return a.get(t)

def downsample_image(img, scale=0.5):
    h, w = img.shape[:2]
    return cv2.resize(cv2.resize(img, (int(w*scale), int(h*scale))), (w, h))

def sharpen_image(img):
    return cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))

def process_cropped_image_gray(img):
    """CLAHE applied directly to the single channel - no LAB conversion needed."""
    out = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(3,3)).apply(img)
    out = cv2.GaussianBlur(out, (7,7), 0)
    out = cv2.bilateralFilter(out, d=5, sigmaColor=75, sigmaSpace=75)
    out = cv2.filter2D(out, -1, np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]))
    return cv2.GaussianBlur(out, (5,5), 0)

def augment_image_gray(img, t):
    if t == "Downsample":          return downsample_image(img)
    if t == "Sharpen":             return sharpen_image(img)
    if t == "TemporalShift":       return img
    if t == "ProcessCroppedImage": return process_cropped_image_gray(img)
    return get_augmentations_gray(t)(image=img)['image']

In [15]:
sample = cv2.imread(f"{GRAY_DIR}/set_01/bat/01.png", cv2.IMREAD_UNCHANGED)
print("input:", sample.shape)

ALL_GRAY = ["Rotate","Translate","Scale","GaussianBlur","GaussNoise","RandomGamma",
            "OpticalDistortion","Perspective","HorizontalFlip","Brightness","Contrast",
            "Downsample","Sharpen","Equalize","ProcessCroppedImage","TemporalShift"]

for t in ALL_GRAY:
    try:
        a = augment_image_gray(sample.copy(), t)
        b = augment_image_gray(sample.copy(), t)
        diff = np.abs(a.astype(int) - sample.astype(int)).mean()
        print(f"{t:22s} shape={str(a.shape):12s} repeat_identical={np.array_equal(a,b)}  mean_change={diff:6.2f}")
    except Exception as e:
        print(f"{t:22s} ERROR: {e}")

input: (80, 112)
Rotate                 shape=(80, 112)    repeat_identical=False  mean_change=  5.26
Translate              shape=(80, 112)    repeat_identical=False  mean_change= 23.17
Scale                  shape=(80, 112)    repeat_identical=False  mean_change= 14.82
GaussianBlur           shape=(80, 112)    repeat_identical=False  mean_change=  3.49
GaussNoise             shape=(80, 112)    repeat_identical=False  mean_change=  9.51
RandomGamma            shape=(80, 112)    repeat_identical=False  mean_change= 29.55
OpticalDistortion      shape=(80, 112)    repeat_identical=False  mean_change=  3.33
Perspective            shape=(80, 112)    repeat_identical=False  mean_change= 13.98
HorizontalFlip         shape=(80, 112)    repeat_identical=True  mean_change= 20.78
Brightness             shape=(80, 112)    repeat_identical=True  mean_change= 75.69
Contrast               shape=(80, 112)    repeat_identical=True  mean_change= 38.29
Downsample             shape=(80, 112)    repeat_id

In [16]:
AUG_GRAY_DIR = os.path.join(WORKSPACE, "03g_frames_augmented_gray")
TARGET_SIZE = (112, 80)     # (width, height)

plan_gray = pd.read_csv(f"{LOGS_DIR}/augmentation_plan_gray_log.csv")

def resize_if_needed_gray(img):
    h, w = img.shape[:2]
    tw, th = TARGET_SIZE
    if (h, w) == (th, tw):
        return img, False
    interp = cv2.INTER_AREA if (h > th or w > tw) else cv2.INTER_LINEAR
    return cv2.resize(img, (tw, th), interpolation=interp), True

def make_one_set_gray(row):
    src_set  = row['source_name']
    aug_name = row['aug_set_name']
    aug_type = row['aug_type']
    resized_count = 0

    for word in WORDS:
        src = os.path.join(GRAY_DIR, src_set, word)
        dst = os.path.join(AUG_GRAY_DIR, aug_name, word)
        os.makedirs(dst, exist_ok=True)
        files = sorted(f for f in os.listdir(src) if f.endswith('.png'))
        for idx, f in enumerate(files, start=1):
            img = cv2.imread(os.path.join(src, f), cv2.IMREAD_UNCHANGED)
            out = augment_image_gray(img, aug_type)
            out, was_resized = resize_if_needed_gray(out)
            if was_resized:
                resized_count += 1
            cv2.imwrite(os.path.join(dst, f"{idx:02d}.png"), out)

    return dict(aug_set_name=aug_name, source_set=src_set, aug_type=aug_type,
                split=row['split'], frames_written=len(WORDS)*60,
                frames_resized=resized_count)

rows = plan_gray.to_dict('records')
start, results, done = datetime.now(), [], 0
with ProcessPoolExecutor(max_workers=10) as ex:
    for fut in as_completed([ex.submit(make_one_set_gray, r) for r in rows]):
        results.append(fut.result())
        done += 1
        if done % 20 == 0:
            print(f"{done}/{len(rows)}  ({(datetime.now()-start).seconds}s)")

gen_gray = pd.DataFrame(results).sort_values(['split','aug_set_name']).reset_index(drop=True)
gen_gray.to_csv(f"{LOGS_DIR}/augmentation_generated_gray_log.csv", index=False)

print("\naugmented sets:", len(gen_gray))
print("per split:", gen_gray.split.value_counts().to_dict())
print("total frames:", gen_gray.frames_written.sum())
print("frames resized:", gen_gray.frames_resized.sum())

chk = cv2.imread(f"{AUG_GRAY_DIR}/aug_train_01/bat/01.png", cv2.IMREAD_UNCHANGED)
print("sample shape:", chk.shape)

20/70  (6s)
40/70  (11s)
60/70  (16s)

augmented sets: 70
per split: {'train': 48, 'test': 11, 'validation': 11}
total frames: 42000
frames resized: 0
sample shape: (80, 112)


In [17]:
GRID_GRAY_DIR    = os.path.join(WORKSPACE, "04g_grids_combined_gray")
RESIZED_GRAY_DIR = os.path.join(WORKSPACE, "05g_grids_resized_gray")
ROWS, COLS = 10, 6

def build_grid_gray(args):
    src_dir, set_label, word = args
    word_path = os.path.join(src_dir, set_label, word)
    frames = sorted(f for f in os.listdir(word_path) if f.endswith('.png'))
    if len(frames) != ROWS * COLS:
        return dict(set_label=set_label, word=word, status=f"WRONG_COUNT_{len(frames)}")

    first = cv2.imread(os.path.join(word_path, frames[0]), cv2.IMREAD_UNCHANGED)
    fh, fw = first.shape[:2]
    grid = np.zeros((fh*ROWS, fw*COLS), dtype=np.uint8)

    for idx, f in enumerate(frames):
        img = cv2.imread(os.path.join(word_path, f), cv2.IMREAD_UNCHANGED)
        r, c = idx // COLS, idx % COLS
        grid[r*fh:(r+1)*fh, c*fw:(c+1)*fw] = img

    os.makedirs(GRID_GRAY_DIR, exist_ok=True)
    cv2.imwrite(os.path.join(GRID_GRAY_DIR, f"{set_label}_{word}.png"), grid)
    return dict(set_label=set_label, word=word, status="OK",
                height=grid.shape[0], width=grid.shape[1])

tasks = [(GRAY_DIR, s, w) for s in sorted(os.listdir(GRAY_DIR)) for w in WORDS]
tasks += [(AUG_GRAY_DIR, s, w) for s in sorted(os.listdir(AUG_GRAY_DIR)) for w in WORDS]
print("grids to build:", len(tasks))

start, results = datetime.now(), []
with ProcessPoolExecutor(max_workers=10) as ex:
    for fut in as_completed([ex.submit(build_grid_gray, t) for t in tasks]):
        r = fut.result()
        results.append(r)
        if r['status'] != "OK":
            print("  PROBLEM", r['set_label'], r['word'], r['status'])

log = pd.DataFrame(results)
log.to_csv(f"{LOGS_DIR}/grid_assembly_gray_log.csv", index=False)
print("built:", (log.status=="OK").sum(), "of", len(log))
print("dimensions:", log[log.status=="OK"].groupby(['height','width']).size().to_dict())

# resize to 224x224
os.makedirs(RESIZED_GRAY_DIR, exist_ok=True)

def resize_gray_one(fname):
    with Image.open(os.path.join(GRID_GRAY_DIR, fname)) as img:
        img.resize((224,224), Image.LANCZOS).save(os.path.join(RESIZED_GRAY_DIR, fname))
    return fname

files = sorted(f for f in os.listdir(GRID_GRAY_DIR) if f.endswith('.png'))
with ProcessPoolExecutor(max_workers=10) as ex:
    list(as_completed([ex.submit(resize_gray_one, f) for f in files]))

print("resized:", len(os.listdir(RESIZED_GRAY_DIR)))
chk = cv2.imread(os.path.join(RESIZED_GRAY_DIR, files[0]), cv2.IMREAD_UNCHANGED)
print("sample shape:", chk.shape)

grids to build: 1300
built: 1300 of 1300
dimensions: {(800, 672): 1300}
resized: 1300
sample shape: (224, 224)


In [18]:
FINAL_GRAY = os.path.join(WORKSPACE, "06g_dataset_gray")
REAL_GRAY  = os.path.join(WORKSPACE, "06g_test_real_only_gray")
SPLIT_FOLDER = {"train":"Train", "validation":"Validation", "test":"Test"}

mapping = pd.read_csv(f"{LOGS_DIR}/final_image_mapping_log.csv")

for _, r in mapping.iterrows():
    dst = os.path.join(FINAL_GRAY, SPLIT_FOLDER[r.split], r.word)
    os.makedirs(dst, exist_ok=True)
    shutil.copy2(os.path.join(RESIZED_GRAY_DIR, r.source_file),
                 os.path.join(dst, r.final_name))

for word in WORDS:
    dst = os.path.join(REAL_GRAY, word)
    os.makedirs(dst, exist_ok=True)
    src = os.path.join(FINAL_GRAY, "Test", word)
    for i in range(1, 10):
        shutil.copy2(os.path.join(src, f"{i:03d}.png"), os.path.join(dst, f"{i:03d}.png"))

print("placed:", len(mapping))
for sp, folder in SPLIT_FOLDER.items():
    p = os.path.join(FINAL_GRAY, folder)
    print(f"{folder}:", {w: len(os.listdir(os.path.join(p, w))) for w in sorted(os.listdir(p))})
print("real-only:", sum(len(os.listdir(os.path.join(REAL_GRAY, w))) for w in WORDS))

placed: 1300
Train: {'bat': 90, 'cup': 90, 'drop': 90, 'eat': 90, 'fish': 90, 'hot': 90, 'jump': 90, 'milk': 90, 'pen': 90, 'red': 90}
Validation: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}
Test: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}
real-only: 90
